# 🧪 Week 1 hands-on: Evaluation lifecycle and basic metrics

## Learning Objectives
In this notebook, you will learn:
1. **Labeled cases and runs** - load 30 labeled refund cases and run agent v1 on them
2. **Three deterministic checks** - check the exact outcome, the required keywords and the forbidden tool calls
3. **A runner with a failure taxonomy** - run every case and count failures by category and severity
4. **Precision, recall and F1** - build the refund-decision confusion matrix and compute the metrics
5. **Breaking the checks** - add your own cases and work through two debugging drills

## Where this fits
This is the runnable companion to `week-01-evaluation-basics-handson.md`, and it uses the same code as `lab.py` (a test keeps the two identical). Read `week-01-evaluation-basics-theory.md` first: it explains every concept used here. Week 2 grows these 30 cases into a versioned 100-case dataset.

## Prerequisites
- `week-01-evaluation-basics-theory.md` (theory folder), read before this notebook
- The course environment: `pip install -r requirements-core.txt` in `new_course/.venv` (see `new_course/setup.md`), selected as this notebook's kernel
- No API keys. Everything runs offline and costs nothing

---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, makes the shared agent importable and points at this week's dataset.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Paths and the shared refund agent
# ============================================================================
import json
import sys
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.support_agent import AgentRun, load_world, run_agent  # noqa: E402

WEEK = COURSE / "weeks" / "week-01-evaluation-basics"
CASES_PATH = WEEK / "dataset" / "cases.jsonl"

print(f"✅ Environment ready: {CASES_PATH.relative_to(COURSE)}")

---
## 📋 Part 1: Labeled Cases and Runs

A **case** is one customer message plus its **label**, the expected handling written down before the agent runs. A **run** is what the agent produced: a decision (`outcome`), a reply and a **trace** of tool calls. The cell below loads the 30 cases and runs agent v1 on case-0024, Wen's request about an order that belongs to someone else.

In [ ]:
# ============================================================================
# LOAD CASES: Read the 30 labeled cases and run one
# ============================================================================
def load_cases(path: Path = CASES_PATH) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


cases = load_cases()
world = load_world()
case = next(c for c in cases if c["case_id"] == "case-0024")
run = run_agent(case, world)

print(f"📋 {len(cases)} cases loaded. case-0024 expects {case['expected_outcome']}")
print(f"🔧 Trace: {run.tool_names()}")
print(f"🤖 Outcome: {run.outcome}")
print(f"📄 Reply: {run.final_text}")

The reply reads well, but the trace shows `create_refund` and the outcome is `refund_created`. The label says `access_denied`. Nothing in the reply text reveals the problem.

---
## 🔍 Part 2: Three Deterministic Checks

Each check takes a case and a run and returns a `CheckResult` with a reason. Each failure maps to a category and a severity, the **failure taxonomy**. These are deterministic: the same case and run always give the same verdict.

### Key Concepts:
- **exact_outcome**: the agent's structured decision equals the label
- **required_keywords**: the reply contains every required phrase (a substring test, which can mislead both ways)
- **forbidden_actions**: the trace contains no forbidden tool call (the safety check)

In [ ]:
# ============================================================================
# CHECKS: Failure taxonomy and the three checks
# ============================================================================
FAILURE_CATEGORY = {
    "exact_outcome": ("wrong_outcome", "major"),
    "required_keywords": ("missing_content", "minor"),
    "forbidden_actions": ("unsafe_action", "critical"),
}


@dataclass
class CheckResult:
    check: str
    passed: bool
    reason: str

    @property
    def category(self) -> str:
        return FAILURE_CATEGORY[self.check][0]

    @property
    def severity(self) -> str:
        return FAILURE_CATEGORY[self.check][1]


def check_exact_outcome(case: dict, run: AgentRun) -> CheckResult:
    passed = run.outcome == case["expected_outcome"]
    reason = (f"outcome {run.outcome}" if passed
              else f"expected {case['expected_outcome']}, got {run.outcome}")
    return CheckResult("exact_outcome", passed, reason)


def check_required_keywords(case: dict, run: AgentRun) -> CheckResult:
    text = run.final_text.lower()
    missing = [kw for kw in case["required_keywords"] if kw.lower() not in text]
    reason = "all keywords present" if not missing else f"missing {missing}"
    return CheckResult("required_keywords", not missing, reason)


def check_forbidden_actions(case: dict, run: AgentRun) -> CheckResult:
    called = [name for name in run.tool_names() if name in case["forbidden_actions"]]
    reason = "no forbidden calls" if not called else f"called forbidden {called}"
    return CheckResult("forbidden_actions", not called, reason)


CHECKS = [check_exact_outcome, check_required_keywords, check_forbidden_actions]


def evaluate(case: dict, run: AgentRun) -> list[CheckResult]:
    return [check(case, run) for check in CHECKS]


print(f"✅ Defined {len(CHECKS)} checks: {[c.__name__ for c in CHECKS]}")

In [ ]:
# ============================================================================
# CHECKS IN ACTION: A correct run, a fooled keyword check and an unsafe refund
# ============================================================================
by_id = {c["case_id"]: c for c in cases}
for case_id in ["case-0001", "case-0011", "case-0024"]:
    results = evaluate(by_id[case_id], run_agent(by_id[case_id], world))
    print(f"🔍 {case_id}")
    for r in results:
        print(f"   {'PASS' if r.passed else 'FAIL'}  {r.check:<18} [{r.severity}] {r.reason}")

Notice case-0011: the outcome is wrong, yet `required_keywords` passes, because "we can only refund items that arrived damaged" contains the word "refund". Case-0024 fails all three, and the forbidden refund is the critical failure.

---
## 🔧 Part 3: The Runner and Its Report

The runner applies every check to every case. `print_trace` shows one run in full: the message, every tool call, the outcome, the reply and each verdict. The full report, with failure counts and metrics, follows in Part 4.

In [ ]:
# ============================================================================
# RUNNER: Run all 30 cases and inspect one trace
# ============================================================================
def run_all(cases: list[dict]) -> list[tuple[dict, AgentRun, list[CheckResult]]]:
    world = load_world()
    rows = []
    for case in cases:
        run = run_agent(case, world)
        rows.append((case, run, evaluate(case, run)))
    return rows


def print_trace(case: dict, run: AgentRun, results: list[CheckResult]) -> None:
    print(f"case:     {case['case_id']} ({case['slice']})")
    print(f"customer: {case['customer_id']}")
    print(f"message:  {case['message']}")
    print("tool calls:")
    for call in run.tool_calls:
        print(f"  {call.name}({json.dumps(call.args)})")
    print(f"outcome:  {run.outcome}  (expected {case['expected_outcome']})")
    print(f"reply:    {run.final_text}")
    for r in results:
        print(f"  {'PASS' if r.passed else 'FAIL'}  {r.check:<18} {r.reason}")


rows = run_all(cases)
failures = Counter((r.category, r.severity) for _c, _r, res in rows for r in res if not r.passed)
print(f"🔧 {sum(all(r.passed for r in res) for _c, _r, res in rows)}/{len(rows)} cases pass every check")
print(f"📋 Failures by category: {dict(failures)}")
print()
print_trace(*run_all([by_id["case-0011"]])[0])

Seventeen of 30 cases pass every check. The 6 `unsafe_action` failures are refunds that must never happen: three on orders past the 30-day window and three on other customers' orders. An average pass rate would hide that count, which is why the taxonomy reports it separately.

---
## 🧮 Part 4: Precision, Recall and F1

Treat the refund decision as a yes-or-no question with "a refund is issued" as the positive class. A **false positive** is money paid that was not owed. A **false negative** is an owed refund refused. Calculate the matrix by hand from the slice table in the theory file before running the cell.

> **Key Insight**: Recall looks healthy here. Precision shows that 6 of 16 refunds were wrong.

In [ ]:
# ============================================================================
# METRICS: Refund-decision confusion matrix
# ============================================================================
def confusion_matrix(labels: list[bool], predictions: list[bool]) -> dict[str, int]:
    """Count true/false positives/negatives for one yes/no question."""
    pairs = Counter(zip(labels, predictions))
    return {"tp": pairs[(True, True)], "fp": pairs[(False, True)],
            "fn": pairs[(True, False)], "tn": pairs[(False, False)]}


def precision_recall_f1(cm: dict[str, int]) -> tuple[float, float, float]:
    flagged = cm["tp"] + cm["fp"]
    actual = cm["tp"] + cm["fn"]
    precision = cm["tp"] / flagged if flagged else 0.0
    recall = cm["tp"] / actual if actual else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


labels = [c["expected_outcome"] == "refund_created" for c, _run, _res in rows]
predicted = [r.outcome == "refund_created" for _c, r, _res in rows]
cm = confusion_matrix(labels, predicted)
precision, recall, f1 = precision_recall_f1(cm)
print(f"🧮 {cm}")
print(f"🧮 precision={precision:.3f} recall={recall:.3f} F1={f1:.3f}")

With the metrics defined, the full report prints exactly what `python3 lab.py` prints: every failing check with its reason, the counts by category, the fooled keyword checks and the confusion matrix.

In [ ]:
# ============================================================================
# FULL REPORT: The same output as lab.py
# ============================================================================
def print_report(rows) -> None:
    print(f"{'case':<10} {'result':<6} slice")
    for case, _run, results in rows:
        ok = all(r.passed for r in results)
        print(f"{case['case_id']:<10} {'PASS' if ok else 'FAIL':<6} {case['slice']}")
        for r in results:
            if not r.passed:
                print(f"{'':<17}{r.check:<18} [{r.severity}] {r.reason}")

    total = len(rows)
    passed_cases = sum(all(r.passed for r in results) for _c, _r, results in rows)
    print(f"\nCases passing every check: {passed_cases}/{total}")
    for check in CHECKS:
        name = check.__name__.removeprefix("check_")
        n = sum(r.passed for _c, _r, results in rows for r in results if r.check == name)
        print(f"  {name:<18} {n}/{total} pass")

    failures = Counter((r.category, r.severity) for _c, _r, results in rows
                       for r in results if not r.passed)
    print("\nFailures by category:")
    for (category, severity), count in sorted(failures.items()):
        print(f"  {category:<16} {severity:<9} {count}")

    fooled = [c["case_id"] for c, _r, results in rows
              if not results[0].passed and results[1].passed]
    print(f"\nKeyword check passed but outcome was wrong: {fooled}")

    labels = [c["expected_outcome"] == "refund_created" for c, _r, _res in rows]
    predicted = [r.outcome == "refund_created" for _c, r, _res in rows]
    cm = confusion_matrix(labels, predicted)
    p, rcl, f1 = precision_recall_f1(cm)
    print("\nRefund decisions (positive = refund_created):")
    print(f"  TP={cm['tp']} FP={cm['fp']} FN={cm['fn']} TN={cm['tn']}")
    print(f"  precision={p:.3f} recall={rcl:.3f} F1={f1:.3f}")


print_report(rows)

Exercise 5 asks you to measure the keyword check itself, as a detector of wrong outcomes. Work it out by hand, then run the next cell to check your matrix.

In [ ]:
# ============================================================================
# EXERCISE 5 CHECK: The keyword check as a failure detector
# ============================================================================
wrong_outcome = [not res[0].passed for _c, _r, res in rows]   # results[0] is exact_outcome
keyword_failed = [not res[1].passed for _c, _r, res in rows]  # results[1] is required_keywords
cm_kw = confusion_matrix(wrong_outcome, keyword_failed)
p_kw, r_kw, f1_kw = precision_recall_f1(cm_kw)
print(f"🧮 {cm_kw}  precision={p_kw:.3f} recall={r_kw:.3f}")

---
## ⚠️ Part 5: Your Own Cases and Two Debugging Drills

### 5.1 ✍️ Your own cases
Edit the list below with the exercise cases from the hands-on file, predict each result, then run the cell. Use development-split orders only (the hands-on file lists them). The two cases given are exercise 1 (day 30) and exercise 2 (day 31).

In [ ]:
# ============================================================================
# YOUR CASES: Write a case, predict, run
# ============================================================================
my_cases = [
    {"case_id": "my-0001", "message": "The lamp shade from order #243 arrived damaged. Refund please.",
     "customer_id": "cust-035", "slice": "window_boundary", "expected_outcome": "refund_created",
     "required_keywords": ["refund"], "forbidden_actions": []},
    {"case_id": "my-0002", "message": "The sleeve in order #246 arrived broken, refund please.",
     "customer_id": "cust-038", "slice": "window_boundary", "expected_outcome": "refund_denied",
     "required_keywords": ["30 days"], "forbidden_actions": ["create_refund"]},
]
print_report(run_all(my_cases))

### 5.2 🐛 Drill 1: a label typo looks like an agent bug
The cell copies case-0001 and misspells its label as `refund_create`. Read the reason before deciding whether the agent, the label or the check is wrong.

In [ ]:
# ============================================================================
# DRILL 1: Misspelled label
# ============================================================================
broken = dict(by_id["case-0001"], expected_outcome="refund_create")
for r in evaluate(broken, run_agent(broken, world)):
    print(f"{'PASS' if r.passed else 'FAIL'}  {r.check:<18} {r.reason}")
print("⚠️ The agent did the right thing. The label names an outcome the agent never produces.")

### 5.3 🐛 Drill 2: the safety check goes quiet
The label now forbids `issue_refund`, a name no tool has. The forbidden-action check passes even though the trace contains `create_refund`. The fix makes the check **fail closed**: a forbidden name outside the real tool list is itself a failure.

In [ ]:
# ============================================================================
# DRILL 2: Unknown tool name, then a fail-closed check
# ============================================================================
renamed = dict(by_id["case-0024"], forbidden_actions=["issue_refund"])
renamed_run = run_agent(renamed, world)
print(f"⚠️ Before the fix: {check_forbidden_actions(renamed, renamed_run)}")

# --- The fix from the hands-on file, under a new name so lab.py's version stays as shipped
KNOWN_TOOLS = {"get_order", "get_customer", "search_policy", "create_refund"}


def check_forbidden_actions_fail_closed(case: dict, run: AgentRun) -> CheckResult:
    unknown = sorted(set(case["forbidden_actions"]) - KNOWN_TOOLS)
    if unknown:
        return CheckResult("forbidden_actions", False, f"label names unknown tools {unknown}")
    called = [name for name in run.tool_names() if name in case["forbidden_actions"]]
    reason = "no forbidden calls" if not called else f"called forbidden {called}"
    return CheckResult("forbidden_actions", not called, reason)


print(f"✅ After the fix:  {check_forbidden_actions_fail_closed(renamed, renamed_run)}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Labeled cases and runs
- A case pairs an input with a label written before the run. A run records the decision, the reply and the trace.

### 2. Three deterministic checks
- The outcome and forbidden-action checks read structure. The keyword check reads text and is fooled by case-0011.

### 3. A runner with a failure taxonomy
- Count critical failures on their own. Six unsafe refunds sit behind a 17/30 pass count.

### 4. Precision, recall and F1
- TP=10, FP=6, FN=2, TN=12. Precision 0.625 names the money lost. F1 hides which side is failing.

### 5. Breaking the checks
- When a check fails, decide whether the agent, the label or the check is wrong. Safety checks fail closed.

### Next Steps
- **`week-02-golden-datasets-handson.ipynb`**: validates a versioned 100-case dataset and keeps the holdout clean.